# 数据理解与标签分布检查

本 notebook 用于确认 Scania APS 数据集的基础结构，包括训练集 / 测试集规模、匿名特征数量、标签分布和成本敏感评估背景。APS 故障样本占比较低，后续模型评估不以 accuracy 为核心，而是围绕漏检成本和维修决策展开。该 notebook 是项目主线的入口，支撑 README 中的业务问题与成本设定部分。

## 1. 数据规模与字段结构

建模前先检查训练集和测试集的规模及字段结构，避免后续清洗、特征构造和建模阶段出现字段错位。

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from scania_aps.config import get_config

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


原始数据保存在 `data/raw/`。本节只读取 official train/test 文件。

In [2]:
cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")

TRAIN_PATH = cfg.train_raw
TEST_PATH = cfg.test_raw
LABEL_COL = cfg.label_column
TARGET_MAPPING = cfg.target_mapping

TRAIN_PATH, TEST_PATH


(WindowsPath('C:/Scania APS/data/raw/aps_failure_training_set.csv'),
 WindowsPath('C:/Scania APS/data/raw/aps_failure_test_set.csv'))

原始 CSV 中的字符串 `na` 表示缺失值，读取时通过 `na_values="na"` 转换为 pandas 缺失值。

In [3]:
train_df = pd.read_csv(TRAIN_PATH, na_values=cfg.missing_value_token)
test_df = pd.read_csv(TEST_PATH, na_values=cfg.missing_value_token)

train_df.head()


,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


下方表格复核 official train/test 的行列规模。

In [4]:
shape_summary = pd.DataFrame(
    {
        "数据集": ["train", "test"],
        "行数": [train_df.shape[0], test_df.shape[0]],
        "列数": [train_df.shape[1], test_df.shape[1]],
    }
)

shape_summary

,数据集,行数,列数
0,train,60000,171
1,test,16000,171


结果显示：训练集包含 60000 行，测试集包含 16000 行；除标签列外，共有 170 个匿名特征。字段匿名化，后续只能进行统计解释

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 171 entries, class to eg_000
dtypes: float64(169), int64(1), object(1)
memory usage: 78.3+ MB


In [6]:
train_df.columns[:10].tolist(), train_df.columns[-10:].tolist()

(['class',
  'aa_000',
  'ab_000',
  'ac_000',
  'ad_000',
  'ae_000',
  'af_000',
  'ag_000',
  'ag_001',
  'ag_002'],
 ['ee_002',
  'ee_003',
  'ee_004',
  'ee_005',
  'ee_006',
  'ee_007',
  'ee_008',
  'ee_009',
  'ef_000',
  'eg_000'])

## 2. 标签分布与类别不平衡

本节确认 APS failure 是否属于少数类问题。`pos` 表示 APS 系统相关故障，`neg` 表示非 APS 系统相关故障。

In [7]:
label_distribution = (
    train_df[LABEL_COL]
    .value_counts(dropna=False)
    .rename_axis(LABEL_COL)
    .reset_index(name="样本数")
)

label_distribution["占比"] = label_distribution["样本数"] / len(train_df)
label_distribution


,class,样本数,占比
0,neg,59000,0.983333
1,pos,1000,0.016667


## 3. FP / FN 成本设定与 target 映射

训练集正类约 1000 / 60000，类别极不平衡，需要把标签分布与 FP/FN 成本设定连接起来。

In [8]:
train_df = train_df.assign(target=train_df[LABEL_COL].map(TARGET_MAPPING))
test_df = test_df.assign(target=test_df[LABEL_COL].map(TARGET_MAPPING))

train_df[[LABEL_COL, "target"]].head()


,class,target
0,neg,0
1,neg,0
2,neg,0
3,neg,0
4,neg,0


In [9]:
train_df["target"].value_counts(dropna=False).sort_index()

target
0    59000
1     1000
Name: count, dtype: int64

## 小结

- 数据由高维匿名工业特征构成，字段只能做统计层面的解释。
- APS failure 是少数类，accuracy 不适合作为核心指标。
- FN 成本是 FP 的 50 倍，后续模型选择必须关注 Recall、F2、PR-AUC、FN 和 total_cost。